# A2A v1.6.4 Tier A staged equilibration
This notebook runs the six frozen Tier A equilibration jobs (two controls × three predeclared seeds) from signed repository bundles using the periodic-distance restraint correction issued after the CUDA diagnostic. It checkpoints to Google Drive. It does **not** run production, unmask candidate labels, or unlock Tier B. Run each seed to completion, then build the aggregate gate report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q --no-cache-dir "openmm[cuda12]==8.6.0" numpy

In [ ]:
from pathlib import Path
import subprocess, openmm
REPO = Path('/content/ECMO-Research-Project')
URL = 'https://github.com/tharranb29-spec/ECMO-Research-Project.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(REPO)], check=True)
platforms = [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
print('OpenMM', openmm.version.short_version, 'platforms:', platforms)
assert 'CUDA' in platforms, 'Select Runtime > Change runtime type > GPU, then reconnect.'
OUT = Path('/content/drive/MyDrive/a2a_md_v16/tier_a_equilibration')
OUT.mkdir(parents=True, exist_ok=True)

## Select one job
Run one system/seed per Colab session. Re-running the same selection resumes from its last completed stage. Do not change the seed list or use only favorable replicas.

In [ ]:
SYSTEM = '5NM4_ZMA_native'  # or 5G53_NECA_miniGs_native_nucleotide_free
SEED = 20260914             # 20260914, 20260915, or 20260916
allowed_systems = ['5NM4_ZMA_native', '5G53_NECA_miniGs_native_nucleotide_free']
allowed_seeds = [20260914, 20260915, 20260916]
assert SYSTEM in allowed_systems and SEED in allowed_seeds
subprocess.run([
    'python', str(REPO/'track3_a2a/run_tier_a_equilibration_v16.py'),
    '--system', SYSTEM, '--seed', str(SEED), '--platform', 'CUDA',
    '--output', str(OUT)
], check=True)

## Build the gate report
Run after all six jobs finish. Production unlocks only when all six full-duration audits pass. A scaled preflight cannot satisfy this gate. Tier B remains locked.

In [ ]:
subprocess.run([
    'python', str(REPO/'track3_a2a/build_tier_a_equilibration_gate_v16.py'),
    '--input', str(OUT)
], check=True)